# Error Analysis & Model Explainability

## Checking GPU Usage

Go to the `Runtime > Change runtime type` menu and check that we're on Python 3 and that the hardware accelerator is set to "GPU".

In [ ]:
!nvidia-smi

## Downloading the Data

In [ ]:
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s landscape

## Importing TensorFlow and the Other Required Libraries

In [ ]:
!pip install shap
import itertools
import pathlib
import typing

import matplotlib.pyplot as plt
import numpy
import PIL.Image
import seaborn
import shap
import sklearn.metrics
import tensorflow as tf
import tensorflow.keras as keras
import tqdm.notebook

## Preparing the Data

To load our data, we'll combine several libraries: [Pillow](https://pillow.readthedocs.io/) (`PIL`), [NumPy](https://numpy.org/) and [TensorFlow](https://www.tensorflow.org/). These libraries will be called from the `get_images` function.

Each image is loaded with PIL, then resized to 150x150; finally, by default, we'll return a dataset shuffled with [`tf.random.shuffle`](https://www.tensorflow.org/api_docs/python/tf/random/shuffle).

In [ ]:
INPUT_SHAPE = (150, 150)


label_names = ["buildings", "forest", "glacier", "mountain", "sea", "street"]
label_to_index = {l: i for i, l in enumerate(label_names)}


def get_images(dir_path: pathlib.Path,
               shuffle: bool = True,
               create_labels: bool = True,
               ) -> typing.Tuple[tf.Tensor, tf.Tensor]:
  images = []
  if create_labels:
    labels = []

  # Iterate over the subfolders of the root: each one matches a
  # class
  for subdir_path in tqdm.notebook.tqdm(
      list(dir_path.iterdir()), desc="Processing folders"):

    dir_name = subdir_path.name

    if create_labels:
      # Assign the right label based on the folder name "labels"
      label = label_to_index.get(dir_name)

    # Add each image of the current label (folder) to our dataset
    for image_path in tqdm.notebook.tqdm(
        list(subdir_path.iterdir()), desc=f"Folder {dir_name}", leave=False):
      # Use PIL to load the image
      images.append(
          numpy.array(PIL.Image.open(image_path).resize(INPUT_SHAPE)))
      if create_labels:
        labels.append(label)

  images = tf.constant(numpy.array(images))
  if create_labels:
    labels = tf.constant(numpy.array(labels))

  if shuffle:
    perm = tf.random.shuffle(tf.range(images.shape[0]))
    images = tf.gather(images, perm)
    if create_labels:
      labels = tf.gather(labels, perm)

  if create_labels:
    return images, labels
  else:
    return images

## Calling `get_images`

In [ ]:
images, labels = get_images(pathlib.Path("landscape") / "seg_train")

In [ ]:
print(f"Images shape:  {images.shape}")
print(f"Labels shape:  {labels.shape}")

seaborn.countplot(x=labels.numpy())
plt.title("Counts of the different labels")
plt.ylabel("Count")
plt.xlabel("Label")
plt.show()

In [ ]:
# Create the grid of subplots. We pass the figsize argument to enlarge
# the figure, which is small by default
f, ax = plt.subplots(5, 5, figsize=(15, 15))

# Pick 25 random indices, without replacement (we don't want to display the
# same image twice)
random_indexes = numpy.random.choice(images.shape[0],
                                     size=(5, 5),
                                     replace=False)

for i in range(5):
  for j in range(5):
    img_index = random_indexes[i, j]
    image = images[img_index]
    label = label_names[labels[img_index]]

    # Display with matplotlib and its imshow function, very handy in computer
    # vision
    ax[i, j].imshow(image)
    ax[i, j].set_title(f"Example {img_index} ({label})")
    ax[i, j].axis('off')

## Finetuning a model

In [ ]:
#@title Definition of a model
base_model = keras.applications.EfficientNetB0(include_top=False,
                                               weights="imagenet",
                                               input_shape=(150, 150, 3))

base_model.trainable = False

transferred_cnn = keras.Sequential(
    [base_model,
     keras.layers.SpatialDropout2D(0.1),
     keras.layers.Conv2D(1024, 1, activation="relu"),
     keras.layers.SpatialDropout2D(0.1),
     keras.layers.Conv2D(256, 1, activation="relu"),
     keras.layers.SpatialDropout2D(0.1),
     keras.layers.Conv2D(64, 1, activation="relu"),
     keras.layers.SpatialDropout2D(0.1),
     keras.layers.Conv2D(16, 1, activation="relu"),
     keras.layers.SpatialDropout2D(0.1),
     keras.layers.Flatten(),
     keras.layers.Dense(6, activation="softmax", kernel_regularizer="l2")],
    name="transferred_cnn")

transferred_cnn.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001),
                        loss="sparse_categorical_crossentropy",
                        metrics=["accuracy"])

transferred_cnn.summary()

In [ ]:
training = transferred_cnn.fit(images,
                               labels,
                               epochs=5,
                               validation_split=0.30,
                               batch_size=512)

## Evaluating the Performance on the Test Set

The `seg_test` folder contains a dataset never seen during training.

We'll use the model's `evaluate(X, y)` method to evaluate the quality of our predictions on this dataset.

In [ ]:
test_images,test_labels = get_images(
    pathlib.Path("landscape") / "seg_test")

In [ ]:
transferred_cnn.evaluate(test_images, test_labels)

## Error Analysis

We display the confusion matrix, then look at misclassified images.

In [ ]:
def predict(model: keras.Model, images: tf.Tensor) -> tf.Tensor:
  return tf.math.argmax(model.predict(images), axis=-1)


def analyze_preds(preds: tf.Tensor, labels: tf.Tensor) -> None:
  confusion_matrix = sklearn.metrics.confusion_matrix(labels, preds)
  seaborn.heatmap(confusion_matrix,
                  cmap="rocket_r",
                  xticklabels=label_names,
                  yticklabels=label_names,
                  annot=True,
                  fmt="d")
  plt.title("Confusion matrix")
  plt.show()

  seaborn.countplot(x=[label_names[x] for x in preds])
  plt.title("Counts of the predicted classes")
  plt.ylabel("Count")
  plt.xlabel("Class")
  plt.show()


test_preds = predict(transferred_cnn, test_images)
analyze_preds(test_preds, test_labels)

In [ ]:
def plot_mistakes(predicted_class: str,
                  true_class: str,
                  images: tf.Tensor,
                  preds: tf.Tensor,
                  labels: tf.Tensor
                  ) -> None:
  print(f"Prediction: {predicted_class}, true class: {true_class}")
  mistakes = images[(preds == label_to_index[predicted_class])
                         & (labels == label_to_index[true_class])]
  random_indexes = numpy.random.choice(mistakes.shape[0],
                                       size=min(mistakes.shape[0], 25),
                                       replace=False)
  grid_indexes = itertools.product(range(5), repeat=2)

  _, ax = plt.subplots(5, 5, figsize=(15, 15))
  for img_index, (i, j) in zip(random_indexes, grid_indexes):
    ax[i, j].imshow(mistakes[img_index])
    ax[i, j].axis("off")
  plt.show()

In [ ]:
# Plot the images predicted as glacier while their label is mountain
plot_mistakes("glacier", "mountain", test_images, test_preds, test_labels)

In [ ]:
# Plot the images predicted as glacier while their label is sea
plot_mistakes("glacier", "sea", test_images, test_preds, test_labels)

In [ ]:
# Plot the images predicted as buildings while their label is sea
plot_mistakes("buildings", "sea", test_images, test_preds, test_labels)

In [ ]:
def evaluate(model: keras.Model, images: tf.Tensor, labels: tf.Tensor) -> None:
  loss, accuracy = model.evaluate(images, labels, verbose=False)
  print(f"Loss: {loss:.2f}, accuracy: {accuracy:.2f}")
  preds = predict(model, images)
  analyze_preds(preds, labels)
  plot_mistakes("glacier", "mountain", images, preds, labels)
  plot_mistakes("glacier", "sea", images, preds, labels)
  plot_mistakes("buildings", "sea", images, preds, labels)


evaluate(transferred_cnn, test_images, test_labels)

## Predicting in "Real" Conditions

The `seg_pred` folder contains unannotated images. We therefore can't properly evaluate the performance on this set.

However, we can display photos and the probabilities our model assigns to each class.

In [ ]:
pred_images = get_images(
    pathlib.Path("landscape") / "seg_pred",
    create_labels=False)

In [ ]:
# Create the grid of subplots. We pass the figsize argument to enlarge
# the figure, which is small by default
_, ax = plt.subplots(10, 5, figsize=(30, 45))

# Pick 25 random indices, without replacement (we don't want to display the
# same image twice)
random_indexes = numpy.random.choice(pred_images.shape[0],
                                     size=(5, 5),
                                     replace=False)

for i in range(5):
  for j in range(5):
    img_index = random_indexes[i, j]
    # Get the image and predict its class
    image = pred_images[img_index]
    probabilities = transferred_cnn.predict(image[None, ...])[0]
    predicted_class = label_names[numpy.argmax(probabilities)]

    # Display with matplotlib and its imshow function, very handy in computer
    # vision
    ax[i * 2, j].imshow(image)
    ax[i * 2, j].set_title(f"Example {img_index}")
    ax[i * 2, j].axis('off')

    # Display the prediction distribution on the row below
    ax[i * 2 + 1, j].bar(label_names, probabilities)

## Model interpretability with SHAP

Use the `shap` library to interpret the model predictions on the first 10 examples of the training dataset (`images`).

In [ ]:
# Your code here

### Solution

In [ ]:
import shap

masker = shap.maskers.Image("blur(32,32)", images[0].shape)

explainer = shap.Explainer(transferred_cnn.predict, masker, output_names=label_names)

shap_values = explainer(
    images[:10].numpy(), max_evals=1_000, batch_size=50, outputs=shap.Explanation.argsort.flip[:3]
)

In [ ]:
shap.image_plot(shap_values)

## SHAP on structured data

Let's retrieve some structured data and train a model:

In [ ]:
#@title Getting and processing structured data
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s ames
import pandas
import numpy
import matplotlib.pyplot as plt
import seaborn
import sklearn.ensemble
import sklearn.model_selection
import scipy.stats
import warnings
warnings.filterwarnings('ignore')

def preprocess(train_file, test_file):
    train_X = pandas.read_csv(train_file, index_col="Id")
    test_X = pandas.read_csv(test_file, index_col="Id")

    train_y = train_X.pop("SalePrice")

    all_X = pandas.concat([train_X, test_X])

    # Fill with median
    cols_1 = ["LotFrontage"]
    all_X[cols_1] = all_X[cols_1].fillna(train_X[cols_1].median())

    # Fill with mode
    cols_2 = ["MSZoning", "Electrical", "KitchenQual", "Exterior1st",
             "Exterior2nd", "SaleType", "Utilities"]
    all_X[cols_2] = all_X[cols_2].fillna(train_X[cols_2].mode().iloc[0, :])

    # Fill with 0
    cols_4 = ["GarageYrBlt", "GarageArea", "GarageCars", "BsmtFinSF1",
              "BsmtFinSF2", "BsmtFullBath", "BsmtHalfBath", "BsmtUnfSF",
              "MasVnrArea", "TotalBsmtSF"]
    all_X[cols_4] = all_X[cols_4].fillna(0)

    # Other fills
    cols_5 = ["Functional"]
    all_X[cols_5] = all_X[cols_5].fillna("Typ")

    all_X = all_X.fillna("NA")

    cols_numerical2label = ['MSSubClass']
    all_X[cols_numerical2label] = all_X[cols_numerical2label].astype(str)

    quality_mapping = dict(NA=0, Po=1, Fa=2, TA=3, Gd=4, Ex=5)
    quality_columns = ["BsmtCond", "BsmtQual", "ExterCond", "ExterQual",
                       "FireplaceQu", "GarageCond", "GarageQual", "HeatingQC",
                       "KitchenQual", "PoolQC"]
    street_mapping = dict(NA=0, Grvl=1, Pave=2)
    bsmt_fin_mapping = dict(NA=0, Unf=1, LwQ=2, Rec=3, BLQ=4, ALQ=5, GLQ=6)

    replace_mapping = dict(
      Alley=street_mapping,
      BsmtExposure=dict(NA=0, No=1, Mn=2, Av=3, Gd=4),
      BsmtFinType1=bsmt_fin_mapping,
      BsmtFinType2=bsmt_fin_mapping,
      Functional=dict(Sal=1, Sev=2, Maj2=3, Maj1=4, Mod=5, Min2=6, Min1=7, Typ=8),
      LandSlope=dict(Sev=1, Mod=2, Gtl=3),
      LotShape=dict(IR3=1, IR2=2, IR1=3, Reg=4),
      PavedDrive=dict(NA=0, N=1, P=2, Y=3),
      Street=dict(Grvl=1, Pave=2),
      Utilities=dict(ELO=1, NoSeWa=2, NoSewr=3, AllPub=4),
    )

    for quality_column in quality_columns:
      replace_mapping[quality_column] = quality_mapping

    all_X.replace(replace_mapping, inplace=True)

    print(f"Number of NAs : {all_X.isnull().sum().sum()}")

    dummies = pandas.get_dummies(all_X)
    return (dummies.iloc[:train_X.shape[0], :],
            train_y,
            dummies.iloc[train_X.shape[0]:, :])

In [ ]:
X, y, _ = preprocess("ames/train.csv", "ames/test.csv")

In [ ]:
rfr = sklearn.ensemble.RandomForestRegressor()
rfr.fit(X, y)

## Global explanation

Explain the marginal gain from each feature on the training dataset using `shap`.

In [ ]:
# Your code here

### Solution

In [ ]:
explainer = shap.TreeExplainer(rfr)
explanation = explainer(X)

In [ ]:
shap.plots.bar(explanation)

## Local explanation

Now display the first example of the dataset and its explanation.

In [ ]:
# Your code here

### Solution

In [ ]:
X.head(1)

In [ ]:
shap.plots.bar(explanation[0])

In [ ]:
shap.plots.force(explainer.expected_value, explanation[0].values, matplotlib=True, feature_names=X.columns)